In [14]:
from keras.datasets import california_housing

(train_data, train_targets), (test_data, test_targets) = (
    california_housing.load_data(version="small")
)

In [5]:
print(train_data.shape)
print(test_data.shape)

(480, 8)
(120, 8)


### Normalize data

In [6]:
mean = train_data.mean(axis=0)
std = train_data.std(axis=0)
x_train = (train_data - mean) / std
x_test = (test_data - mean) / std

In [7]:
x_train[0]

array([-0.1761497, -1.5940031, -2.4240775,  3.4239438,  2.5898013,
        2.9189088,  2.8009396,  0.5019599], dtype=float32)

In [8]:
y_train = train_targets / 100000
y_test = test_targets / 100000

In [9]:
def get_model():
    model = keras.Sequential(
        [
            layers.Dense(64, activation="relu"),
            layers.Dense(64, activation="relu"),
            layers.Dense(1)
        ]
    )
    model.compile(
        optimizer="adam",
        loss="mean_squared_error",
        metrics=["mean_absolute_error"],
    )
    return model

## K-fold cross-validation

In [16]:
import keras
from keras import layers

In [ ]:
k = 4
num_val_samples = len(x_train) // k
num_epochs = 50
all_scores = []
for i in range(k):
    # Prepares the validation data: data from partition #k
    fold_x_val = x_train[i * num_val_samples : (i + 1) * num_val_samples]
    fold_y_val = y_train[i * num_val_samples : (i + 1) * num_val_samples]
    # Prepares the training data: data from all other partitions
    fold_x_train = np.concatenate(
        [x_train[: i * num_val_samples], x_train[(i + 1) * num_val_samples :]],
        axis=0,
    )
    fold_y_train = np.concatenate(
        [y_train[: i * num_val_samples], y_train[(i + 1) * num_val_samples :]],
        axis=0,
    )
    # Builds the Keras model (already compiled)
    model = get_model()
    # Trains the model
    model.fit(
        fold_x_train,
        fold_y_train,
        epochs=num_epochs,
        batch_size=16,
        verbose=0,
    )
    # Evaluates the model on the validation data
    scores = model.evaluate(fold_x_val, fold_y_val, verbose=0)
    val_loss, val_mae = scores
    all_scores.append(val_mae)

Processing fold #1
Processing fold #2
Processing fold #3
Processing fold #4


In [18]:
[round(value, 3) for value in all_scores]

[0.306, 0.288, 0.23, 0.31]

In [19]:
round(np.mean(all_scores), 3)

np.float64(0.283)

In [20]:
predictions = model.predict(x_test)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step


In [23]:
[predictions[i] for i in range(5)]


[array([2.1314294], dtype=float32),
 array([1.9557835], dtype=float32),
 array([1.2637854], dtype=float32),
 array([1.7610095], dtype=float32),
 array([2.2723994], dtype=float32)]

In [24]:
[y_test[i] for i in range(5)]

[np.float32(2.188),
 np.float32(2.184),
 np.float32(0.938),
 np.float32(1.734),
 np.float32(2.297)]

In [26]:
[((y_test[i] - predictions[i]) * 100) / y_test[i] for i in range(5)]

[array([2.5854905], dtype=float32),
 array([10.449475], dtype=float32),
 array([-34.73191], dtype=float32),
 array([-1.5576406], dtype=float32),
 array([1.0709841], dtype=float32)]